# 05 - Stage 3: Hardened Multi-Signal Attestation Check

**Goal:** Detect forged Stage 2a and Stage 2b models that pass naive public-fingerprint cosine checks by combining:
1. Public refusal-direction fingerprint cosine (`fingerprint_cosine`)
2. Refusal-circuit top-component Jaccard overlap (`circuit_overlap`)
3. Secret multilingual probe refusal rate (`probe_refusal_rate`)
4. SHA-256 cryptographic commitment verification (`verify_commitment`)

In [ ]:
!git clone https://github.com/mathcrypto/forged-fingerprints.git /content/forged-fingerprints 2>/dev/null || (cd /content/forged-fingerprints && git pull)
%cd /content/forged-fingerprints
!pip uninstall -y torchao -q
!pip install -q -e . peft accelerate

In [ ]:
import os
from huggingface_hub import login, snapshot_download

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')
login(token=token) if token else login()

snapshot_download('mathcrypto/refusal-fingerprints', local_dir='data/reference')
snapshot_download('mathcrypto/gemma-2-9b-it-stage2a-fingerprint-preservation', local_dir='models/stage2a')
snapshot_download('mathcrypto/gemma-2-9b-it-stage2b-fingerprint-loss', local_dir='models/stage2b')

In [ ]:
!python -m scripts.run_hardened_check \
  --model_path models/stage2a \
  --ref_path data/reference \
  --commitment_path data/reference/commitment.json \
  --output_json results/stage3_stage2a_hardened.json

!python -m scripts.run_hardened_check \
  --model_path models/stage2b \
  --ref_path data/reference \
  --commitment_path data/reference/commitment.json \
  --output_json results/stage3_stage2b_hardened.json